# Emotion Detection from Text - Exploratory Analysis

This notebook provides an exploratory analysis of the emotion detection project.

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# NLP imports
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud

# ML imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Custom imports
import sys
sys.path.append('../src')
from emotion_detector import EmotionDetector

# Plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. Load and Explore Sample Data

In [ ]:
# Sample emotional text data for exploration
sample_data = [
    ("I'm so happy and excited about this new opportunity!", "joy"),
    ("This is the best day of my life!", "joy"),
    ("I feel fantastic and full of energy!", "joy"),
    ("Celebrating another wonderful achievement!", "joy"),
    ("Life is beautiful and I'm grateful!", "joy"),
    
    ("I feel so sad and lonely today", "sadness"),
    ("This news makes me deeply upset", "sadness"),
    ("I'm crying because I miss them so much", "sadness"),
    ("Feeling depressed and hopeless", "sadness"),
    ("This tragic event breaks my heart", "sadness"),
    
    ("I'm furious about what they did!", "anger"),
    ("This makes me so angry and frustrated!", "anger"),
    ("I can't believe how rude they were!", "anger"),
    ("This injustice makes my blood boil!", "anger"),
    ("I'm absolutely livid right now!", "anger"),
    
    ("That horror movie scared me to death!", "fear"),
    ("I'm terrified of what might happen", "fear"),
    ("The thought of failure frightens me", "fear"),
    ("I'm anxious and worried about tomorrow", "fear"),
    ("This situation makes me very nervous", "fear"),
    
    ("Wow, I never saw that coming!", "surprise"),
    ("What an unexpected turn of events!", "surprise"),
    ("I'm shocked by this revelation!", "surprise"),
    ("That was completely out of the blue!", "surprise"),
    ("I can't believe my eyes!", "surprise"),
    
    ("This food tastes absolutely disgusting", "disgust"),
    ("That smell makes me nauseous", "disgust"),
    ("I find their behavior repulsive", "disgust"),
    ("This is gross and unappetizing", "disgust"),
    ("The sight of it makes me sick", "disgust"),
    
    ("The weather is okay today", "neutral"),
    ("I need to go to the store", "neutral"),
    ("The meeting is scheduled for 3 PM", "neutral"),
    ("Please pass me the salt", "neutral"),
    ("The report is due on Friday", "neutral"),
]

# Create DataFrame
df = pd.DataFrame(sample_data, columns=['text', 'emotion'])
print(f"Dataset shape: {df.shape}")
print("\nFirst few rows:")
df.head()

## 3. Data Exploration and Visualization

In [ ]:
# Emotion distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
emotion_counts = df['emotion'].value_counts()
emotion_counts.plot(kind='bar', color='skyblue')
plt.title('Emotion Distribution')
plt.xlabel('Emotions')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
plt.pie(emotion_counts.values, labels=emotion_counts.index, autopct='%1.1f%%')
plt.title('Emotion Distribution (Pie Chart)')

plt.tight_layout()
plt.show()

print("Emotion counts:")
print(emotion_counts)

In [ ]:
# Text length analysis
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df.boxplot(column='text_length', by='emotion', ax=plt.gca())
plt.title('Text Length by Emotion')
plt.xlabel('Emotion')
plt.ylabel('Character Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
df.boxplot(column='word_count', by='emotion', ax=plt.gca())
plt.title('Word Count by Emotion')
plt.xlabel('Emotion')
plt.ylabel('Word Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Summary statistics
print("Text statistics by emotion:")
print(df.groupby('emotion')[['text_length', 'word_count']].describe())

## 4. Text Preprocessing Analysis

In [ ]:
# Initialize emotion detector for preprocessing
detector = EmotionDetector()

# Apply preprocessing
df['processed_text'] = df['text'].apply(detector.preprocess_text)
df['processed_word_count'] = df['processed_text'].str.split().str.len()

# Compare original vs processed
print("Sample preprocessing results:")
for i in range(5):
    print(f"\nOriginal: {df.iloc[i]['text']}")
    print(f"Processed: {df.iloc[i]['processed_text']}")
    print("-" * 50)

In [ ]:
# Word frequency analysis by emotion
def get_top_words_by_emotion(df, emotion, n=10):
    texts = df[df['emotion'] == emotion]['processed_text'].str.cat(sep=' ')
    words = texts.split()
    return Counter(words).most_common(n)

# Create word clouds for each emotion
emotions = df['emotion'].unique()
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, emotion in enumerate(emotions):
    if i < len(axes):
        # Get text for this emotion
        text = df[df['emotion'] == emotion]['processed_text'].str.cat(sep=' ')
        
        if text.strip():  # Only create wordcloud if there's text
            wordcloud = WordCloud(width=400, height=200, 
                                background_color='white',
                                colormap='viridis').generate(text)
            
            axes[i].imshow(wordcloud, interpolation='bilinear')
            axes[i].set_title(f'{emotion.title()} Words', fontsize=14)
            axes[i].axis('off')

# Hide unused subplots
for i in range(len(emotions), len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 5. Model Training and Comparison

In [ ]:
# Prepare data for modeling
X = df['processed_text']
y = df['emotion']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"\nTraining emotion distribution:")
print(y_train.value_counts())

In [ ]:
# Compare different models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Naive Bayes': MultinomialNB()
}

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train_tfidf, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"Accuracy: {accuracy:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Plot model comparison
plt.figure(figsize=(10, 6))
model_names = list(results.keys())
accuracies = list(results.values())

plt.bar(model_names, accuracies, color=['skyblue', 'lightcoral', 'lightgreen'])
plt.title('Model Accuracy Comparison')
plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

# Add accuracy values on bars
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Get the best performing model (usually Logistic Regression for text)
best_model = models['Logistic Regression']
feature_names = vectorizer.get_feature_names_out()

# Get top features for each emotion class
emotions_list = best_model.classes_
coefficients = best_model.coef_

plt.figure(figsize=(15, 10))

for i, emotion in enumerate(emotions_list):
    # Get top 10 features for this emotion
    top_indices = np.argsort(coefficients[i])[-10:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    top_scores = coefficients[i][top_indices]
    
    plt.subplot(2, 4, i+1)
    plt.barh(range(len(top_features)), top_scores)
    plt.yticks(range(len(top_features)), top_features)
    plt.xlabel('Feature Importance')
    plt.title(f'Top Features for {emotion.title()}')
    plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

## 7. Interactive Testing

In [ ]:
# Test the model with custom text
def test_emotion_detection(text):
    """
    Test emotion detection on custom text
    """
    # Preprocess text
    processed_text = detector.preprocess_text(text)
    
    # Transform with vectorizer
    text_tfidf = vectorizer.transform([processed_text])
    
    # Get prediction and probabilities
    prediction = best_model.predict(text_tfidf)[0]
    probabilities = best_model.predict_proba(text_tfidf)[0]
    
    print(f"Original text: {text}")
    print(f"Processed text: {processed_text}")
    print(f"Predicted emotion: {prediction}")
    print("\nProbability breakdown:")
    
    for emotion, prob in zip(emotions_list, probabilities):
        print(f"  {emotion}: {prob:.3f}")
    
    return prediction, probabilities

# Test examples
test_texts = [
    "I'm absolutely thrilled about this amazing news!",
    "This situation makes me feel really down and hopeless.",
    "I can't believe how irritating this is!",
    "That sudden noise really startled me.",
    "What an incredible and unexpected surprise!",
    "This smells absolutely terrible.",
    "The meeting is scheduled for next Tuesday at 2 PM."
]

print("Testing emotion detection on sample texts:")
print("=" * 60)

for text in test_texts:
    prediction, probs = test_emotion_detection(text)
    print("\n" + "-" * 60)


## 8. Model Performance Visualization

In [ ]:
# Confusion Matrix
y_pred_best = best_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, y_pred_best, labels=emotions_list)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=emotions_list, yticklabels=emotions_list)
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
plt.tight_layout()
plt.show()

# Classification report as heatmap
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred_best, output_dict=True)
report_df = pd.DataFrame(report).iloc[:-1, :-1].T  # Exclude support and avg rows

plt.figure(figsize=(8, 6))
sns.heatmap(report_df, annot=True, cmap='viridis', fmt='.2f')
plt.title('Classification Metrics by Emotion')
plt.tight_layout()
plt.show()

## 9. Conclusions and Next Steps

### Key Findings:
1. **Model Performance**: Logistic Regression typically performs well for text classification tasks
2. **Feature Importance**: Certain words are strongly associated with specific emotions
3. **Data Balance**: The dataset is balanced across emotions, which is good for training

### Next Steps for Improvement:
1. **Larger Dataset**: Collect more diverse and larger datasets
2. **Advanced Models**: Try BERT, RoBERTa, or other transformer models
3. **Data Augmentation**: Use techniques to expand the training data
4. **Hyperparameter Tuning**: Optimize model parameters
5. **Cross-Validation**: Use k-fold cross-validation for better evaluation
6. **Feature Engineering**: Try different text representations (word2vec, FastText, etc.)
